# Chapter 14 &mdash; Procedure vs. Algorithm, and the First Impossibility Result

**Concept 2 of the Chapter 14 decomposition:** *Procedure vs. Algorithm, and the First Impossibility Result*

An algorithm is a procedure that halts on <i>all</i> inputs &mdash; and no machine can tell which procedures are algorithms.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter14/Concept-Procedure-Vs-Algorithm/Concept-Procedure-Vs-Algorithm.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
from jove.Def_TM         import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Two words that are not synonyms:

* a **procedure** is any TM &mdash; it may halt or loop;
* an **algorithm** is a TM that **halts on every input**.

Every algorithm is a procedure; the converse fails. And here is the first impossibility
result, stated before it is proved:

> **No procedure decides whether a given procedure is an algorithm.**

This is the halting problem wearing different clothes, and it is why "is this function
total?" is not a question your compiler can answer. The rest of the chapter builds the
vocabulary to prove it.

## 2. Definitions

### Three machines: two algorithms and a procedure

In [ ]:
Alg1 = md2mc('''TM
I : 0 ; 0 , R -> F
I : 1 ; 1 , R -> D
I : . ; . , R -> D
''')
Alg2 = md2mc('''TM
I : 0 ; 1 , R -> I
I : 1 ; 0 , R -> I
I : . ; . , S -> F
''')
Proc = md2mc('''TM
I : 0 ; 0 , R -> F
I : 1 ; 1 , R -> L
I : . ; . , R -> L
L : 0 ; 0 , R -> L
L : 1 ; 1 , R -> L
L : . ; . , R -> L
''')

# --- thin wrappers over Jove's TM runner --------------------------------
def tm_accepts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return any(cfg[0] in T["F"] for cfg, _ in halts)

def tm_halts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return len(halts) > 0

### Testing totality -- and why the test can only ever be partial

In [ ]:
from itertools import product
def halts_on_all(T, upto=5, fuel=200, sigma='01'):
    for k in range(1, upto + 1):
        for p in product(sigma, repeat=k):
            if not tm_halts(T, ''.join(p), fuel=fuel):
                return False, ''.join(p)
    return True, None

## 3. Tests

Two of the three halt on everything we try.

In [ ]:
for name, T in [('Alg1', Alg1), ('Alg2', Alg2), ('Proc', Proc)]:
    ok, wit = halts_on_all(T)
    print("  %-6s halts on all tested inputs? %-6s %s"
          % (name, ok, ('witness ' + repr(wit)) if wit else ''))
assert halts_on_all(Alg1)[0] and halts_on_all(Alg2)[0]
assert not halts_on_all(Proc)[0]

But **the test is a search, not a decision.** It can only ever find a witness.

In [ ]:
print("halts_on_all returns False only when it FINDS a diverging input.")
print("It returns True only because it gave up after a finite sweep.")
print()
print("Enlarging the sweep never turns it into a decision procedure:")
for upto in [3, 5, 7]:
    ok, _ = halts_on_all(Alg1, upto=upto)
    print("   tested up to length %d : %s -- still not a proof" % (upto, ok))

The vocabulary, precisely.

In [ ]:
print("procedure : any TM.  May halt, may loop.")
print("algorithm : a TM that halts on EVERY input.")
print()
print("recognizer (semi-decider) : implements a procedure")
print("decider                   : implements an algorithm")

And the result the chapter is heading for.

In [ ]:
print("CLAIM  no procedure decides, of an arbitrary procedure, whether it")
print("       is an algorithm.")
print()
print("If one existed you could ask it about a machine that loops exactly")
print("when some other machine fails to halt -- and you would have solved")
print("the halting problem.  Chapter 15 makes the reduction precise.")

## 4. Exercises


1. Is every DFA an algorithm? Every PDA? Every TM?
2. Give a procedure that is an algorithm on one alphabet but not on another.
3. Why does "test it on a million inputs" not settle totality?

In [ ]:
# Your work for the exercises above.